# Notes on *An Empirical Model of Large-Batch Training*

## Core idea

Large batch training is **not** always better.

The paper argues that there is a **largest useful batch size**, often approximated by the **critical batch size**.  
Below this size, increasing batch size helps a lot.  
Above this size, increasing batch size gives **diminishing returns**.

---

## Two different kinds of efficiency

### 1. Time efficiency
This means:
- fewer optimizer steps to reach a target loss
- less wall-clock training time if hardware can parallelize well

So a larger batch can be good for **time efficiency**.

### 2. Compute / data efficiency
This means:
- how many examples or tokens are consumed to reach a target loss
- how much total compute is spent to get there

So a larger batch can be **bad for compute efficiency** if it uses many more samples but does not reduce the number of training steps by much.

---

## Linear region

When batch size is **below** the critical batch size:

- the gradient estimate is still noisy
- increasing batch size reduces gradient noise
- training progress improves almost proportionally

In this regime, doubling batch size can nearly double useful progress per step.

---

## Plateau region

When batch size is **above** the critical batch size:

- the gradient is already estimated well enough
- making the batch larger does not improve the update much
- training may still be stable
- but the extra examples per step are mostly wasted

So this region is where **time efficiency improves only a little**, while **compute/data efficiency gets worse**.

---

## Important distinction

### Largest batch that fits on GPU
This is a **hardware memory limit**.

### Largest batch that is stable
This is an **optimization stability limit**.

### Largest batch that is actually useful
This is the **critical batch size** or something close to it.

These are **not the same thing**.

A batch can:
- fit on GPU
- be stable
- and still be inefficient

---

## Why warmup and learning rate schedules matter

Large batch size itself does **not automatically mean a larger gradient update**.

What large batch mainly does is:
- reduce gradient noise
- make the minibatch gradient closer to the full-batch gradient

But in practice, when people increase batch size, they often also increase the learning rate.  
That is where warmup and learning rate schedules become important:
- to avoid instability early in training
- to make large-batch training work well in practice

---

## Critical batch size and gradient noise scale

A simple approximation is:

**critical batch size ∝ gradient variance / squared mean gradient norm**

So:

- if gradient signal gets smaller faster than variance shrinks, critical batch size increases
- if variance shrinks faster, critical batch size decreases
- if both shrink at similar rates, critical batch size stays about the same

---

## Why critical batch size often increases later in training

Later in training:

- the true gradient signal usually becomes smaller
- minibatch noise does not always shrink as fast

So the signal-to-noise ratio gets worse.

That means a small batch becomes less reliable, and a larger batch is needed to estimate the true gradient direction well enough.

---

## My summary

The paper is really about a tradeoff:

- **small batch** = better compute/data efficiency, but slower in wall-clock time
- **large batch** = better parallelism and time efficiency, but only up to a point
- **too large batch** = diminishing returns and wasted compute/tokens

So the goal is **not to use the largest possible batch**, but to use the **largest useful batch**.

# Notes on *An Empirical Model of Large-Batch Training*

## 1. Setup

Let the loss over the data distribution be

$$
L(\theta) = \mathbb{E}_{x \sim \rho}[L_x(\theta)].
$$

The true gradient is

$$
G(\theta) = \nabla L(\theta).
$$

A minibatch gradient estimate with batch size $B$ is

$$
G_{\text{est}}(\theta) = \frac{1}{B}\sum_{i=1}^{B} \nabla_\theta L_{x_i}(\theta), \qquad x_i \sim \rho.
$$

Its expectation and covariance are

$$
\mathbb{E}[G_{\text{est}}(\theta)] = G(\theta),
$$

$$
\operatorname{Cov}(G_{\text{est}}(\theta)) = \frac{1}{B}\Sigma(\theta),
$$

where the per-example gradient covariance is

$$
\Sigma(\theta)
= \operatorname{Cov}_{x \sim \rho}(\nabla_\theta L_x(\theta))
= \mathbb{E}_{x \sim \rho}\!\left[(\nabla_\theta L_x(\theta))(\nabla_\theta L_x(\theta))^\top\right]
- G(\theta)G(\theta)^\top.
$$

So larger batch size reduces gradient noise like $1/B$.

---

## 2. Local quadratic model of one update

Suppose we update parameters by

$$
\theta \leftarrow \theta - \epsilon V.
$$

Using a second-order Taylor expansion of the true loss around $\theta$,

$$
L(\theta - \epsilon V)
\approx
L(\theta) - \epsilon G^\top V + \frac{1}{2}\epsilon^2 V^\top H V,
$$

where $H$ is the Hessian of the true loss at $\theta$.

If we had the true gradient, we would choose $V = G$.

In practice, we only have the noisy minibatch estimate, so set

$$
V = G_{\text{est}}.
$$

Taking expectation over minibatches gives

$$
\mathbb{E}[L(\theta - \epsilon G_{\text{est}})]
=
L(\theta)
-
\epsilon \|G\|^2
+
\frac{1}{2}\epsilon^2
\left(
G^\top H G + \frac{\operatorname{tr}(H\Sigma)}{B}
\right).
$$

This is the key equation linking batch size to optimization progress.

---

## 3. Optimal learning rate from the local model

Differentiate the expected loss with respect to $\epsilon$ and set to zero:

$$
\frac{d}{d\epsilon}\mathbb{E}[L(\theta - \epsilon G_{\text{est}})] = 0.
$$

This gives the optimal local step size

$$
\epsilon_{\text{opt}}(B)
=
\frac{\|G\|^2}
{G^\top H G + \operatorname{tr}(H\Sigma)/B}.
$$

Now define

$$
\epsilon_{\max} = \frac{\|G\|^2}{G^\top H G}
$$

and

$$
B_{\text{noise}} = \frac{\operatorname{tr}(H\Sigma)}{G^\top H G}.
$$

Then the optimal learning rate becomes

$$
\epsilon_{\text{opt}}(B)
=
\frac{\epsilon_{\max}}{1 + B_{\text{noise}}/B}.
$$

This is the paper’s local prediction for how the optimal learning rate changes with batch size.

---

## 4. Optimal loss improvement and the linear-to-plateau transition

Plugging $\epsilon_{\text{opt}}(B)$ back in gives the optimal expected improvement in loss

$$
\Delta L_{\text{opt}}(B)
=
\frac{\Delta L_{\max}}{1 + B_{\text{noise}}/B},
$$

where

$$
\Delta L_{\max}
=
\frac{1}{2}\frac{\|G\|^4}{G^\top H G}.
$$

This equation explains the two regimes:

### Small-batch regime: $B \ll B_{\text{noise}}$

Then

$$
\Delta L_{\text{opt}}(B)
\approx
\Delta L_{\max}\frac{B}{B_{\text{noise}}},
$$

so improvement grows roughly linearly with $B$.

### Large-batch regime: $B \gg B_{\text{noise}}$

Then

$$
\Delta L_{\text{opt}}(B) \approx \Delta L_{\max},
$$

so further increases in batch size barely help.

### Turning point

The turning point is around

$$
B \approx B_{\text{noise}}.
$$

This is the point where training speed drops to about 50% of the maximum possible local improvement.

---

## 5. Simplified noise scale

The exact noise scale uses the Hessian:

$$
B_{\text{noise}} = \frac{\operatorname{tr}(H\Sigma)}{G^\top H G}.
$$

That is expensive to estimate.

If we make the simplifying assumption that the optimization is well-conditioned, meaning roughly

$$
H \propto I,
$$

then the noise scale reduces to

$$
B_{\text{simple}} = \frac{\operatorname{tr}(\Sigma)}{\|G\|^2}.
$$

This is much easier to measure.

The paper also gives a useful interpretation:

$$
\frac{\mathbb{E}[\|G_{\text{est}} - G\|^2]}{\|G\|^2}
=
\frac{1}{B}\frac{\operatorname{tr}(\Sigma)}{\|G\|^2}
=
\frac{B_{\text{simple}}}{B}.
$$

So $B_{\text{simple}}/B$ is a natural **normalized gradient-noise level**.

A convenient shorthand is:

- **inverse-noise-adjusted quality** $\propto B_{\text{simple}}/B$
- **signal-to-noise style proxy** $\propto B/B_{\text{simple}}$

The paper directly gives the normalized gradient error formula above; the “SNR proxy” wording is just an interpretation of that equation.

---

## 6. Data efficiency vs time efficiency

The paper distinguishes between:

- **time efficiency**: how many optimizer steps are needed to reach a target level of performance
- **compute/data efficiency**: how many total training examples are consumed to reach that same target

Let

- $S$ = actual number of optimizer steps to reach a target performance
- $S_{\min}$ = minimum possible number of optimizer steps
- $E$ = actual number of training examples processed
- $E_{\min}$ = minimum possible number of training examples processed

Then the paper predicts the tradeoff

$$
\frac{S}{S_{\min}} - 1
=
\left(
\frac{E}{E_{\min}} - 1
\right)^{-1}.
$$

Equivalently,

$$
\left(\frac{S}{S_{\min}} - 1\right)\left(\frac{E}{E_{\min}} - 1\right) = 1.
$$

This is the hyperbolic tradeoff curve.

---

## 7. Critical batch size

The empirical critical batch size is defined as

$$
B_{\text{crit}} = \frac{E_{\min}}{S_{\min}}.
$$

This gives a natural compromise point between time efficiency and compute efficiency.

Why?

At

$$
B = B_{\text{crit}},
$$

the two sides of the tradeoff equation are both equal to $1$, so

$$
S = 2S_{\min}, \qquad E = 2E_{\min}.
$$

So training at $B_{\text{crit}}$ uses only twice the minimum possible steps and twice the minimum possible examples.

The paper’s core prediction is

$$
B_{\text{crit}} \approx B_{\text{noise}},
$$

and in practice often approximately

$$
B_{\text{crit}} \approx B_{\text{simple}}
$$

up to order of magnitude.

---

## 8. Assumptions and caveats

The theory is useful, but it depends on several assumptions:

### 1. Short-horizon bias
The derivation is local. It tells us what is best for the **next step**, not necessarily for the full training trajectory.

### 2. Poor conditioning
If the loss landscape is ill-conditioned, the local model can be inaccurate, especially for exact progress per step.

### 3. Simplified noise scale
Using $B_{\text{simple}}$ instead of $B_{\text{noise}}$ assumes the Hessian is well-conditioned enough that Hessian weighting does not change things too much.

### 4. Learning-rate tuning
The derivation assumes the learning rate is tuned near the local optimum. Poorly tuned learning rates can distort the observed scaling with batch size.

### 5. Quadratic approximation
The Taylor expansion is only second order. Higher-order effects are ignored.

### 6. Generalization
The theory is about optimizing training loss, not directly about test performance or generalization.

### 7. Warmup is not part of the core derivation
The paper explicitly says its treatment ignores issues like the relationship between early and late training and the necessity of a warm-up period.

---

## 9. Predictions and how the equations explain them

### A. Decaying learning rate changes the measured noise scale

Appendix C defines a training temperature

$$
T(\epsilon, B) = \frac{\epsilon}{\epsilon_{\max}(B)}.
$$

For pure SGD in the small-batch regime, this is approximately

$$
T \approx \frac{\epsilon}{B}.
$$

The paper then argues that in equilibrium

$$
B_{\text{noise}} \propto B_{\text{simple}} \propto \frac{1}{T}.
$$

So:

- decrease $\epsilon$ at fixed $B$ $\Rightarrow$ lower temperature $\Rightarrow$ larger measured noise scale
- increase $B$ and $\epsilon$ together appropriately $\Rightarrow$ roughly preserve the noise scale

This is why learning-rate tuning matters when measuring or comparing noise scales.

---

### B. Weak dependence on model size

From the simplified expression

$$
B_{\text{simple}} = \frac{\operatorname{tr}(\Sigma)}{\|G\|^2},
$$

there is no explicit model-size term.

So at **fixed loss**, the theory predicts only weak direct dependence on model size.

Larger models can still show larger noise scales in practice, but mainly because they often reach lower loss.

---

### C. Growth over training

From

$$
B_{\text{simple}} = \frac{\operatorname{tr}(\Sigma)}{\|G\|^2},
$$

if $\operatorname{tr}(\Sigma)$ stays roughly constant while $\|G\|^2$ shrinks as training progresses, then

$$
B_{\text{simple}} \uparrow.
$$

This explains why the critical batch size often grows over training.

---

### D. Larger for more difficult tasks

Harder tasks should tend to have less correlated examples or more diverse gradients, which increases gradient variance.

That means larger $\operatorname{tr}(\Sigma)$, so

$$
B_{\text{simple}} = \frac{\operatorname{tr}(\Sigma)}{\|G\|^2}
$$

should tend to be larger.

This is more of a qualitative prediction than a strict theorem.

---

## 10. Practical interpretation

The key idea is **not** “use the largest batch that fits in memory.”

There are three different notions:

### Largest batch that fits on GPU
A hardware memory limit.

### Largest batch that is stable
An optimization stability limit.

### Largest batch that is useful
A statistical/optimization limit, approximately near $B_{\text{crit}}$.

These are not the same thing.

---

## 11. How to measure the noise scale in practice

Appendix A.1 gives a practical estimator.

For a batch of size $B$,

$$
\mathbb{E}[\|G_{\text{est}}\|^2]
=
\|G\|^2 + \frac{1}{B}\operatorname{tr}(\Sigma).
$$

If we measure gradient norms for two batch sizes, $B_{\text{small}}$ and $B_{\text{big}}$, then we can estimate

$$
\widehat{\|G\|^2}
=
\frac{
B_{\text{big}}\|G_{B_{\text{big}}}\|^2
-
B_{\text{small}}\|G_{B_{\text{small}}}\|^2
}{
B_{\text{big}} - B_{\text{small}}
},
$$

and

$$
\widehat{\operatorname{tr}(\Sigma)}
=
\frac{
\|G_{B_{\text{small}}}\|^2
-
\|G_{B_{\text{big}}}\|^2
}{
1/B_{\text{small}} - 1/B_{\text{big}}
}.
$$

Then estimate

$$
\widehat{B_{\text{simple}}}
=
\frac{
\widehat{\operatorname{tr}(\Sigma)}
}{
\widehat{\|G\|^2}
}.
$$

The paper warns that the ratio itself is not unbiased, so these estimates should be averaged over many batches or smoothed over time.

---

## 12. Small-transformer experiment plan

### Goal
Measure how the noise scale evolves during training and test whether it predicts the largest useful batch size.

### Hypotheses
1. $B_{\text{simple}}$ increases over training.
2. The empirically useful batch size should be near $B_{\text{simple}}$ up to order of magnitude.
3. Learning-rate decay will change the measured noise scale unless training temperature is controlled.
4. At fixed loss, changing model size should have only weak direct effect on the noise scale.

### What to log at checkpoints
At selected checkpoints over training, log:

- training loss
- validation loss
- $\widehat{\|G\|^2}$
- $\widehat{\operatorname{tr}(\Sigma)}$
- $\widehat{B_{\text{simple}}}$
- tokens/sec
- optimizer steps/sec

### How to compare with actual useful batch size
Run separate training sweeps over fixed batch sizes and retune the learning rate for each one.

For each batch size, train to the same target validation loss and record:

- $S(B)$ = total optimizer steps to target
- $E(B)$ = total examples or tokens processed to target

Then estimate:

$$
S_{\min} = \min_B S(B), \qquad E_{\min} = \min_B E(B),
$$

and

$$
B_{\text{crit}} \approx \frac{E_{\min}}{S_{\min}}.
$$

Then compare this to the average measured $\widehat{B_{\text{simple}}}$ over the relevant phase of training.

### A practical single-GPU version
If I do not have multi-GPU data parallelism, I can still approximate the measurement:

1. Save a checkpoint.
2. Sample multiple disjoint microbatches of size $b$.
3. Compute the gradient on each microbatch without stepping the optimizer.
4. Average several of those gradients to emulate a larger batch $kb$.
5. Use the two-batch formulas above with $B_{\text{small}} = b$ and $B_{\text{big}} = kb$.
6. Repeat many times and average.

### Plots to make
1. $\widehat{B_{\text{simple}}}$ vs training step
2. $\widehat{B_{\text{simple}}}$ vs validation loss
3. $\widehat{\|G\|^2}$ vs training step
4. $\widehat{\operatorname{tr}(\Sigma)}$ vs training step
5. $S(B)$ vs $B$
6. $E(B)$ vs $B$
7. Estimated $B_{\text{crit}}$ vs average $\widehat{B_{\text{simple}}}$

### Important controls
- Compare checkpoints at similar loss, not just similar step number.
- Retune learning rate for each batch size.
- Keep data order, sequence length, optimizer, and architecture fixed when possible.
- Be careful with learning-rate decay, because it changes the temperature and can inflate the measured noise scale.
- Average over many measurements, because the ratio estimator is noisy.

---

## 13. My distilled takeaway

The paper’s theory says:

- gradient noise falls like $1/B$
- optimal progress initially improves with batch size
- beyond a turning point near the noise scale, larger batches mostly waste compute
- the largest useful batch size is therefore set by the gradient noise scale, not by GPU memory alone

So the real goal is not maximum batch size, but approximately the **largest useful batch size**.